# A3.3 — Router Sorularını Başkalaştırma (Paraphrase) — Colab

Bu notebook **Google Colab'da**, ücretsiz T4 GPU ile çalışacak şekilde yazıldı.
Yerelde (kendi bilgisayarında) çalıştırma — GPU yok, çok yavaş olur.

**Ne yapıyor:** `router_sorulari.jsonl`'daki her soruyu, orta boy açık kaynak bir
LLM'e (Qwen2.5-7B-Instruct) vererek 2-3 farklı başkalaştırılmış (paraphrase)
versiyonunu üretiyor. Amaç: router modelinin eğitiminde şablonların ürettiği
kalıpların dışına çıkan, daha doğal/çeşitli soru varyantları olsun.

**Neden ayrı bir model?** Kişi B'nin kuracağı Qwen2.5-**1.5B** üretimde
(routing + gerekçe yazımı) çalışacak küçük model. Bu notebook'taki
Qwen2.5-**7B** ise yalnızca **eğitim verisi hazırlarken bir kerelik**
kullanılan, çok daha güçlü bir model — küçük modeli büyük modelin ürettiği
kaliteli örneklerle eğitiyoruz (yaygın bir teknik: "büyük öğretmen, küçük
öğrenci").

**Sıra:**
1. Runtime → Değiştir → Donanım hızlandırıcı → **T4 GPU** seç (aşağıda anlatılıyor)
2. `router_sorulari.jsonl` dosyasını yükle (yerelde `uv run python -m
   training.build_dataset --sadece-router` ile üretilmişti)
3. Önce **100 örnekle** dene, süreyi ölç
4. Süre makulse tam veri setinde çalıştır, sonucu indir


## 1. Ortamı hazırla

In [ ]:
!nvidia-smi


In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece


## 2. Veri dosyasını yükle

Aşağıdaki hücre bir dosya seçme penceresi açacak — bilgisayarındaki `router_sorulari.jsonl` dosyasını seç.

In [ ]:
from google.colab import files

yuklenen = files.upload()
girdi_dosyasi = next(iter(yuklenen.keys()))
print("Yüklenen dosya:", girdi_dosyasi)


## 3. Modeli yükle (4-bit, T4'e sığması için)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ADI = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ADI)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ADI,
    quantization_config=bnb_config,
    device_map="auto",
)
print("Model yüklendi.")


## 4. Başkalaştırma (paraphrase) fonksiyonu

In [ ]:
import json

PROMPT_SABLONU = '''Aşağıdaki Türkçe soruyu, AYNI ANLAMI koruyarak {n} farklı
şekilde yeniden yaz. Her varyant farklı bir üslupla olsun (biri daha resmi,
biri daha günlük konuşma dili). Yalnızca yeniden yazılmış soruları, her biri
ayrı satırda ve başında numara olmadan yaz. Başka hiçbir şey yazma.

Soru: {soru}'''


def paraphrase_uret(soru: str, n: int = 2, max_new_tokens: int = 200) -> list[str]:
    mesajlar = [{"role": "user", "content": PROMPT_SABLONU.format(soru=soru, n=n)}]
    girdi = tokenizer.apply_chat_template(
        mesajlar, tokenize=False, add_generation_prompt=True
    )
    girdi_id = tokenizer(girdi, return_tensors="pt").to(model.device)

    with torch.no_grad():
        cikti = model.generate(
            **girdi_id,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    metin = tokenizer.decode(
        cikti[0][girdi_id["input_ids"].shape[1] :], skip_special_tokens=True
    )
    satirlar = [s.strip(" -•\t") for s in metin.split("\n") if s.strip()]
    return satirlar[:n]


## 5. Önce 100 örnekle dene — süreyi ölç

Roadmap'in kuralı: **önce küçük örnekle maliyeti/süreyi ölç, sonra tam veri setine geç.** Burada 'maliyet' parasal değil (Colab ücretsiz), ama T4'ün oturum süresi sınırlı — tam veri setinin kaç saat süreceğini önceden bilmek gerekiyor.

In [ ]:
import time

with open(girdi_dosyasi, encoding="utf-8") as f:
    tum_sorular = [json.loads(satir) for satir in f]

print("Toplam soru sayısı:", len(tum_sorular))

ornek_100 = tum_sorular[:100]

t0 = time.time()
sonuclar_100 = []
for kayit in ornek_100:
    varyantlar = paraphrase_uret(kayit["soru"], n=2)
    sonuclar_100.append({**kayit, "varyantlar": varyantlar})
t1 = time.time()

sure_100 = t1 - t0
print(f"100 soru için süre: {sure_100:.1f} sn ({sure_100 / 100:.2f} sn/soru)")
print(f"Tüm veri seti ({len(tum_sorular)} soru) için tahmini süre: "
      f"{sure_100 / 100 * len(tum_sorular) / 60:.1f} dakika")

print()
print("--- İlk 3 örnek ---")
for s in sonuclar_100[:3]:
    print("Orijinal:", s["soru"])
    print("Varyantlar:", s["varyantlar"])
    print()


## 6. Kalite kontrolü (elle)

**Durup yukarıdaki örnekleri oku.** Varyantlar doğal mı, anlam korunuyor mu, aynı `arac`/`parametreler` hâlâ geçerli mi? Doğal değilse `PROMPT_SABLONU`'nu güncelleyip 4. hücreyi yeniden çalıştır, tekrar dene.

## 7. Tam veri setinde çalıştır

Yukarıdaki tahmini süre makulse (birkaç saatin altındaysa) bu hücreyi çalıştır. Uzun sürebilir — Colab'ın ücretsiz oturumu ~12 saatte veya ~90 dk hareketsizlikte kesiliyor, o yüzden ara ara sekmeyle etkileşimde kal (rastgele bir hücreye tıklamak yeterli).

In [ ]:
ARA_KAYIT_ARALIGI = 500
CIKTI_DOSYASI = "router_sorulari_parafraz.jsonl"

with open(CIKTI_DOSYASI, "w", encoding="utf-8") as f:
    for i, kayit in enumerate(tum_sorular):
        varyantlar = paraphrase_uret(kayit["soru"], n=2)
        f.write(json.dumps({**kayit, "varyantlar": varyantlar}, ensure_ascii=False) + "\n")

        if (i + 1) % ARA_KAYIT_ARALIGI == 0:
            print(f"{i + 1}/{len(tum_sorular)} işlendi...")

print("Bitti:", CIKTI_DOSYASI)


## 8. Sonucu indir

In [ ]:
from google.colab import files

files.download(CIKTI_DOSYASI)


## Sonrası

İndirdiğin `router_sorulari_parafraz.jsonl` dosyasını projede
`data/egitim/router_sorulari_parafraz.jsonl` olarak kaydet — bir sonraki
oturumda bu dosyayı birlikte gözden geçiririz (A3.5'in bir parçası olarak
train/val/test bölmesine dahil edilecek).